In [ ]:
# Parameters — injected by CI (do not edit manually)
ci_target = "ephemeral_ci"
prod_state_path = "abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/Files/ci-artifacts/prod-state"
repo_url = "https://github.com/ORG/REPO.git"
repo_branch = "BRANCH"
github_app_id = "GH-APP-ID"
github_installation_id = "GH-APP-INSTALLATION-ID"
github_pem_secret = "GH-APP-PEM"
vault_url = "https://KV-NAME.vault.azure.net/"
lakehouse_name = "LAKEHOUSE_NAME"
lakehouse_id = "LAKEHOUSE_ID"
workspace_id = "WORKSPACE_ID"
workspace_name = "WORKSPACE_NAME"
schema_name = "dbo"
gate = "2"
head_sha = "HEAD_SHA"

In [ ]:
# Download prod-state manifest from OneLake (ABFSS → local path)
# No-op when prod_state_path is already a local path (e.g. greenfield ./prod-state).
if prod_state_path.startswith('abfss://'):
    import os
    # The provision job uploads the prod-state manifest to the ephemeral lakehouse
    # at Files/ci-artifacts/prod-state/manifest.json before this notebook runs.
    # Use the FUSE-mounted path instead of /tmp/ — /tmp/ on the Jupyter driver is
    # not visible inside run_dbt_job's execution context (Livy subprocess isolation).
    local_dir = '/lakehouse/default/Files/ci-artifacts/prod-state'
    manifest_file = f'{local_dir}/manifest.json'
    print(f"Checking prod-state manifest at: {manifest_file}")
    if not os.path.exists(manifest_file):
        raise RuntimeError(
            f"Prod-state manifest not found at {manifest_file}. "
            f"Expected to be uploaded by the provision job (source: {prod_state_path})."
        )
    local_prod_state_path = local_dir
    print(f"Prod-state manifest verified at {manifest_file}")
else:
    local_prod_state_path = prod_state_path

In [ ]:
# Command assembly
_deps  = "dbt deps"
_clone = f"dbt clone --select state:modified+ --exclude config.materialized:view --state {local_prod_state_path} --target-path target/clone --profiles-dir .github/profiles --target {ci_target}"
_build = f"dbt run --select state:modified+ --defer --state {local_prod_state_path} --target-path target/build --profiles-dir .github/profiles --target {ci_target}"
_unit  = f"dbt test --select state:modified+ --select test_type:unit --state {local_prod_state_path} --target-path target/unit --profiles-dir .github/profiles --target {ci_target}"
_data  = f"dbt test --select state:modified+ --exclude test_type:unit --store-failures --state {local_prod_state_path} --target-path target/data --profiles-dir .github/profiles --target {ci_target}"


# CI gate commands (dbt deps runs once per gate)
gate2_commands = [_deps, _clone, _build]
gate3_commands = [_deps, _unit]
gate4_commands = [_deps, _data]

In [ ]:
!pip install vd-dbt-fabricspark==1.9.15 -q

In [ ]:
from typing import Any, Callable

ExecutorFn = Callable[[str], list]

ColumnInfo = dict  # {"name": str, "dtype": str, "nullable": bool}

VALUE_DELTA_MATERIALIZATIONS: frozenset[str] = frozenset({"table", "incremental", "snapshot"})
SKIP_MATERIALIZATIONS: frozenset[str] = frozenset({"ephemeral"})


# ── Pure core ──────────────────────────────────────────────────────────────────

def compute_schema_delta(
    prod_columns: list[ColumnInfo],
    ci_columns: list[ColumnInfo],
) -> dict:
    """Return schema delta: added, removed, renamed, type_changed, nullability_flipped."""
    prod_by_name = {c["name"]: c for c in prod_columns}
    ci_by_name = {c["name"]: c for c in ci_columns}
    prod_names = set(prod_by_name)
    ci_names = set(ci_by_name)

    type_changed = []
    nullability_flipped = []
    for name in sorted(prod_names & ci_names):
        p, c = prod_by_name[name], ci_by_name[name]
        if p.get("dtype") != c.get("dtype"):
            type_changed.append({
                "column": name,
                "prod_dtype": p.get("dtype"),
                "ci_dtype": c.get("dtype"),
            })
        if bool(p.get("nullable")) != bool(c.get("nullable")):
            nullability_flipped.append({
                "column": name,
                "prod_nullable": p.get("nullable"),
                "ci_nullable": c.get("nullable"),
            })

    return {
        "added": sorted(ci_names - prod_names),
        "removed": sorted(prod_names - ci_names),
        "renamed": [],
        "type_changed": type_changed,
        "nullability_flipped": nullability_flipped,
    }


def compute_row_count_delta(prod_count: int, ci_count: int) -> dict:
    """Return {"prod": int, "pr": int, "delta": int}. delta = pr - prod."""
    return {"prod": prod_count, "pr": ci_count, "delta": ci_count - prod_count}


def normalize_unique_key(raw: Any) -> list[str] | None:
    """Normalise manifest unique_key to list[str] or None.

    manifest.json stores unique_key as a string (single-column) or list (composite).
    Returns None when key is absent, empty, or an unexpected type.
    """
    if raw is None:
        return None
    if isinstance(raw, str):
        return [raw] if raw else None
    if isinstance(raw, list):
        return raw if raw else None
    return None


def _diff_where(pa: str, ca: str, keys: list[str], common_columns: list[str]) -> str:
    """Generate WHERE clause to detect differing rows via FULL OUTER JOIN.

    Detects:
    - Rows added to CI (missing in prod, detected by prod key IS NULL)
    - Rows deleted from prod (missing in CI, detected by CI key IS NULL)
    - Rows with value changes in non-key columns (detected by IS DISTINCT FROM)
    """
    non_key = [c for c in common_columns if c not in set(keys)]
    null_side = f"({pa}.`{keys[0]}` IS NULL OR {ca}.`{keys[0]}` IS NULL)"
    if not non_key:
        return null_side
    col_diffs = " OR ".join(
        f"({pa}.`{c}` IS DISTINCT FROM {ca}.`{c}`)" for c in non_key
    )
    return f"({null_side} OR ({col_diffs}))"


def build_value_delta_count_sql(
    prod_table: str,
    ci_table: str,
    keys: list[str],
    common_columns: list[str],
) -> str:
    """Return a COUNT query that counts differing rows via key-based FULL OUTER JOIN.

    Joins prod and CI tables on all key columns, then counts rows where:
    - Either side is NULL (row added/deleted), or
    - Non-key column values differ (IS DISTINCT FROM).
    """
    join_on = " AND ".join(f"p.`{k}` = c.`{k}`" for k in keys)
    where = _diff_where("p", "c", keys, common_columns)
    return (
        f"SELECT COUNT(*) AS rows_with_diffs "
        f"FROM {prod_table} p "
        f"FULL OUTER JOIN {ci_table} c ON {join_on} "
        f"WHERE {where}"
    )


# ── I/O shell ─────────────────────────────────────────────────────────────────

def _extract_scalar(result: list) -> int:
    """Extract a single integer from a one-row executor result.

    Handles both dict rows ({"rows_with_diffs": n}) and tuple rows ((n,)).
    """
    if not result:
        return 0
    row = result[0]
    if isinstance(row, dict):
        return int(next(iter(row.values())))
    if isinstance(row, (list, tuple)):
        return int(row[0])
    return int(row)


def execute_value_delta(
    executor: "ExecutorFn",
    prod_table: str,
    ci_table: str,
    keys: list[str],
    common_columns: list[str],
) -> dict:
    """Run the key-based JOIN count query via executor; return value_delta dict.

    executor(sql) must return a list with one row containing the count.
    Use a mocked executor in tests; pass a real Spark SQL executor in the notebook.
    """
    sql = build_value_delta_count_sql(prod_table, ci_table, keys, common_columns)
    rows_with_diffs = _extract_scalar(executor(sql))
    return {
        "sampled_rows": min(rows_with_diffs, 10000),
        "rows_with_diffs": rows_with_diffs,
        "skipped_no_unique_key": False,
    }


# ── Artifact-level orchestration ──────────────────────────────────────────────

def compute_artifact_diff(
    artifact: dict,
    prod_columns: list[ColumnInfo],
    ci_columns: list[ColumnInfo],
    prod_count: int,
    ci_count: int,
    value_delta: dict | None,
) -> dict:
    """Assemble the per-artifact result dict from pre-fetched values.

    The calling notebook cell (VD-1974) is responsible for:
    - skipping ephemeral materializations (check SKIP_MATERIALIZATIONS before calling)
    - deciding whether to compute value delta (check VALUE_DELTA_MATERIALIZATIONS)
    - fetching prod_columns, ci_columns, prod_count, ci_count from the database
    - passing value_delta=None for view/materialized_view (demo trim)
    - passing {"skipped_no_unique_key": True, ...} when model has no unique_key

    brand-new artifact (pre_existing_in_prod: false) → baseline: null, no deltas.
    """
    unique_id = artifact["unique_id"]
    name = artifact.get("name", "")
    materialized = artifact.get("materialized", "")

    if not artifact.get("pre_existing_in_prod", True):
        return {
            "unique_id": unique_id,
            "name": name,
            "materialized": materialized,
            "baseline": None,
            "schema_delta": None,
            "row_count_delta": None,
            "value_delta": None,
        }

    return {
        "unique_id": unique_id,
        "name": name,
        "materialized": materialized,
        "baseline": name,
        "schema_delta": compute_schema_delta(prod_columns, ci_columns),
        "row_count_delta": compute_row_count_delta(prod_count, ci_count),
        "value_delta": value_delta,
    }


# ── Gate signal + result builder ───────────────────────────────────────────────

def gate_5_overall_status(artifacts: list[dict]) -> str:
    """Return 'pass' or 'fail'.

    Pass: all brand-new (all baselines null) OR no non-empty diff across artifacts.
    Fail: any artifact with non-empty schema, row-count, or value diff.
    Skipped value delta (skipped_no_unique_key: true or value_delta: null) does NOT fail.
    """
    for a in artifacts:
        if a.get("baseline") is None:
            continue  # brand-new — auto-pass

        schema = a.get("schema_delta") or {}
        if any(schema.get(k) for k in ("added", "removed", "renamed", "type_changed", "nullability_flipped")):
            return "fail"

        row = a.get("row_count_delta") or {}
        if row.get("delta", 0) != 0:
            return "fail"

        value = a.get("value_delta") or {}
        if value and not value.get("skipped_no_unique_key", False) and value.get("rows_with_diffs", 0) > 0:
            return "fail"

    return "pass"


def build_gate_5_result(head_sha: str, artifacts: list[dict]) -> dict:
    """Assemble the gate-5.json dict (design spec §9.2.2)."""
    return {
        "gate": "5",
        "head_sha": head_sha,
        "overall_status": gate_5_overall_status(artifacts),
        "artifacts": artifacts,
    }


In [ ]:
from dbt.adapters.fabricspark.notebook import (
    run_dbt_job,
    DbtJobConfig,
    RepoConfig,
    ConnectionConfig,
)

In [ ]:
# CI gate functions
import json
import os
os.environ["SESSION_ID_FILE"] = f"Files/ci-artifacts/{head_sha}/livy-session-id.txt"
os.environ["PR_NUMBER"] = workspace_name.rsplit("_", 1)[-1] if workspace_name else "manual"

# vd-dbt-fabricspark always clones to this directory
_PROJECT_DIR = "/tmp/dbt_project"

_repo = RepoConfig(
    url=repo_url,
    branch=repo_branch,
    github_app_id=github_app_id,
    github_installation_id=github_installation_id,
    github_pem_secret=github_pem_secret,
    vault_url=vault_url,
)
_conn = ConnectionConfig(
    lakehouse_name=lakehouse_name,
    lakehouse_id=lakehouse_id,
    workspace_id=workspace_id,
    workspace_name=workspace_name,
    schema_name=schema_name,
)

def _make_job(commands):
    return DbtJobConfig(command=commands, repo=_repo, connection=_conn)

def _load_run_results(subdir):
    """Load run_results.json from the dbt project's target directory."""
    path = os.path.join(_PROJECT_DIR, subdir, "run_results.json")
    return json.load(open(path)) if os.path.exists(path) else {"results": []}

def _write_gate_result(gate_num, result_dict):
    """Upload gate result to OneLake via DFS HTTP API — no FUSE mount dependency."""
    import urllib.request
    data = json.dumps(result_dict, indent=2).encode('utf-8')
    remote = f"Files/ci-artifacts/gate-{gate_num}/{head_sha}/gate-{gate_num}.json"
    base = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lakehouse_id}/{remote}"
    token = notebookutils.credentials.getToken('storage')  # noqa: F821
    auth = {'Authorization': f'Bearer {token}'}
    def _req(url, method, body=b'', extra=None):
        req = urllib.request.Request(url, data=body, method=method, headers={**auth, **(extra or {})})
        with urllib.request.urlopen(req) as r:
            return r.status
    _req(f"{base}?resource=file&overwrite=true", 'PUT')
    _req(f"{base}?action=append&position=0", 'PATCH', data, {'Content-Length': str(len(data))})
    _req(f"{base}?action=flush&position={len(data)}", 'PATCH')
    print(f"Gate {gate_num} result written to OneLake: {remote}")

def _model_rows(run_results):
    return [
        {
            "name": r.get("unique_id", "").split(".")[-1],
            "status": r.get("status", ""),
            "duration_seconds": r.get("execution_time", 0.0),
            "error_message": (r.get("message") or "")[:500] or None,
        }
        for r in run_results.get("results", [])
    ]

def _step_status(models):
    return "pass" if all(m["status"] in ("success", "pass") for m in models) else "fail"


def check_gate_2():
    try:
        run_dbt_job(_make_job(gate2_commands))
    except Exception:
        pass  # dbt/Spark failures — results written to disk before session loss
    clone_models = _model_rows(_load_run_results("target/clone"))
    build_models = _model_rows(_load_run_results("target/build"))
    clone_status = _step_status(clone_models)
    build_status = _step_status(build_models)
    overall = "pass" if clone_status == "pass" and build_status == "pass" else "fail"
    _write_gate_result("2", {
        "gate": "2", "head_sha": head_sha, "overall_status": overall,
        "clone": {"status": clone_status, "models": clone_models},
        "build": {"status": build_status, "models": build_models},
    })


def check_gate_3():
    try:
        run_dbt_job(_make_job(gate3_commands))
    except Exception:
        pass
    run_results = _load_run_results("target/unit")
    counts = {"pass": 0, "fail": 0, "error": 0, "skip": 0}
    failures = []
    for r in run_results.get("results", []):
        s = r.get("status", "")
        if s in ("pass", "success"):
            counts["pass"] += 1
        elif s == "fail":
            counts["fail"] += 1
            failures.append({"name": r.get("unique_id", ""), "status": "fail", "message": (r.get("message") or "")[:500]})
        elif s == "skip":
            counts["skip"] += 1
        else:
            counts["error"] += 1
            failures.append({"name": r.get("unique_id", ""), "status": "error", "message": (r.get("message") or "")[:500]})
    overall = "fail" if (counts["fail"] or counts["error"]) else "pass"
    truncated = len(failures) > 10
    _write_gate_result("3", {
        "gate": "3", "head_sha": head_sha, "overall_status": overall,
        "total": sum(counts.values()), "counts": counts,
        "failures": failures[:10], "truncated": truncated,
    })


def check_gate_4():
    try:
        run_dbt_job(_make_job(gate4_commands))
    except Exception:
        pass
    run_results = _load_run_results("target/data")
    tests_out = [
        {
            "name": r.get("unique_id", "").split(".")[-1],
            "model": (r.get("unique_id", "").split(".") + [""])[2] if len(r.get("unique_id", "").split(".")) > 2 else "",
            "status": r.get("status", ""),
            "duration_seconds": r.get("execution_time", 0.0),
            "failures_count": r.get("failures", 0) or 0,
            "store_failures_table": f"dbt_test__audit.{r.get('unique_id', '').split('.')[-1]}" if r.get("status") in ("fail", "error") else None,
            "message": (r.get("message") or "")[:500] or None,
        }
        for r in run_results.get("results", [])
    ]
    overall = "fail" if any(t["status"] in ("fail", "error") for t in tests_out) else "pass"
    _write_gate_result("4", {
        "gate": "4", "head_sha": head_sha, "overall_status": overall,
        "tests": tests_out,
    })


def check_gate_5():
    # Read deployment manifest
    manifest_raw = notebookutils.fs.head(  # noqa: F821
        f"Files/ci-artifacts/deployment-manifest-{head_sha}.json"
    )
    deployment_manifest = json.loads(manifest_raw)
    artifacts_in = deployment_manifest.get("artifacts", [])

    # Read dbt manifest for unique_key info
    dbt_manifest_raw = notebookutils.fs.head("Files/ci-artifacts/dbt-manifest.json")  # noqa: F821
    dbt_nodes = json.loads(dbt_manifest_raw).get("nodes", {})

    # Prod workspace/lakehouse names for Spark 4-part notation.
    # TODO: inject PROD_WORKSPACE_NAME and PROD_LAKEHOUSE_NAME via generate_ci_notebook.py
    prod_workspace_name = os.environ.get("PROD_WORKSPACE_NAME", "")
    prod_lakehouse_name = os.environ.get("PROD_LAKEHOUSE_NAME", "")

    def _spark_executor(sql):
        return [row.asDict() for row in spark.sql(sql).collect()]  # noqa: F821

    artifacts_out = []
    for artifact in artifacts_in:
        mat = artifact.get("materialized", "")
        if mat in SKIP_MATERIALIZATIONS:
            continue
        if not artifact.get("pre_existing_in_prod", True):
            artifacts_out.append(compute_artifact_diff(artifact, [], [], 0, 0, None))
            continue

        name = artifact.get("name", "")
        table_name = name.split(".")[-1]
        ci_ref = f"`{lakehouse_name}`.`{schema_name}`.`{table_name}`"
        prod_ref = f"`{prod_workspace_name}`.`{prod_lakehouse_name}`.`dbo`.`{table_name}`"

        ci_schema = spark.table(ci_ref).schema  # noqa: F821
        prod_schema = spark.table(prod_ref).schema  # noqa: F821
        ci_cols = [{"name": f.name, "dtype": str(f.dataType), "nullable": f.nullable}
                   for f in ci_schema]
        prod_cols = [{"name": f.name, "dtype": str(f.dataType), "nullable": f.nullable}
                     for f in prod_schema]
        ci_count = spark.table(ci_ref).count()  # noqa: F821
        prod_count = spark.table(prod_ref).count()  # noqa: F821

        value_delta = None
        if mat in VALUE_DELTA_MATERIALIZATIONS:
            raw_key = (dbt_nodes.get(artifact["unique_id"]) or {}).get("config", {}).get("unique_key")
            keys = normalize_unique_key(raw_key)
            if keys:
                common = sorted(
                    {c["name"] for c in ci_cols} & {c["name"] for c in prod_cols}
                )
                value_delta = execute_value_delta(
                    _spark_executor, prod_ref, ci_ref, keys, common
                )
            else:
                value_delta = {
                    "sampled_rows": 0,
                    "rows_with_diffs": 0,
                    "skipped_no_unique_key": True,
                }

        artifacts_out.append(
            compute_artifact_diff(artifact, prod_cols, ci_cols, prod_count, ci_count, value_delta)
        )

    _write_gate_result("5", build_gate_5_result(head_sha, artifacts_out))


In [ ]:
# CI dispatch
gate_fns = {"2": check_gate_2, "3": check_gate_3, "4": check_gate_4, "5": check_gate_5}
gate_fns[gate]()